In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName('ecom_exloratory_analsis').getOrCreate()

In [2]:
sc = spark.sparkContext

In [3]:
spark

In [4]:
customers = spark.read.option('header', 'true').csv('data/input/unprocessed/olist_customers_dataset.csv')

In [5]:
customers.show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                   09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                   01151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                   08775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [36]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



from bronze to silver

# Customers Table

In [6]:
customers.describe().show()

+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|  count|               99441|               99441|                   99441|              99441|         99441|
|   mean|                null|                null|       35137.47458291851|               null|          null|
| stddev|                null|                null|       29797.93899620613|               null|          null|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                   01003|abadia dos dourados|            AC|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|            TO|
+-------+--------------------+--------------------+------------------------+-------------------+--------

In [7]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [8]:
nulls_count = customers.select([count(when(col(c).isNull(), c)).alias(c) for c in customers.columns])
nulls_count.show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [9]:
nulls_count = customers.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in customers.columns])
nulls_count.show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [10]:
customers = customers.withColumnRenamed('customer_zip_code_prefix', 'customer_zip_code').drop('customer_unique_id')

customers.show()

+--------------------+-----------------+--------------------+--------------+
|         customer_id|customer_zip_code|       customer_city|customer_state|
+--------------------+-----------------+--------------------+--------------+
|06b8999e2fba1a1fb...|            14409|              franca|            SP|
|18955e83d337fd6b2...|            09790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|            01151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|            08775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|            13056|            campinas|            SP|
|879864dab9bc30475...|            89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|            04534|           sao paulo|            SP|
|5e274e7a0c3809e14...|            35182|             timoteo|            MG|
|5adf08e34b2e99398...|            81560|            curitiba|            PR|
|4b7139f34592b3a31...|            30575|      belo horizonte|            MG|

In [37]:
customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



# orders_items Table

In [11]:
order_items = spark.read.option('header', 'true').csv('data/input/unprocessed/olist_order_items_dataset.csv')

order_items.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.90|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.90|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.00|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.90|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [12]:
order_items = order_items.drop('order_item_id', 'shipping_limit_date')

In [13]:
order_items.show(3)

+--------------------+--------------------+--------------------+------+-------------+
|            order_id|          product_id|           seller_id| price|freight_value|
+--------------------+--------------------+--------------------+------+-------------+
|00010242fe8c5a6d1...|4244733e06e7ecb49...|48436dade18ac8b2b...| 58.90|        13.29|
|00018f77f2f0320c5...|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|239.90|        19.93|
|000229ec398224ef6...|c777355d18b72b67a...|5b51032eddd242adc...|199.00|        17.87|
+--------------------+--------------------+--------------------+------+-------------+
only showing top 3 rows



In [14]:
order_items = order_items.select('order_id', 'product_id', 'seller_id', col('price').cast(FloatType()), col('freight_value').cast(FloatType()))

In [15]:
order_items_nulls = order_items.select([count(when(col(c).isNull(), c)).alias(c) for c in order_items.columns])

order_items_nulls.show()

+--------+----------+---------+-----+-------------+
|order_id|product_id|seller_id|price|freight_value|
+--------+----------+---------+-----+-------------+
|       0|         0|        0|    0|            0|
+--------+----------+---------+-----+-------------+



In [38]:
order_items.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- price: float (nullable = true)
 |-- freight_value: float (nullable = true)



# Orders table

In [16]:
orders = spark.read.option('header', 'true').csv('data/input/unprocessed/olist_orders_dataset.csv')

orders.show(3)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [17]:
orders = (orders.select('order_id', 'customer_id', col('order_purchase_timestamp').alias('purchase_timestamp').cast(TimestampType())))

orders.show(3)

+--------------------+--------------------+-------------------+
|            order_id|         customer_id| purchase_timestamp|
+--------------------+--------------------+-------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|2017-10-02 10:56:33|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|2018-07-24 20:41:37|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|2018-08-08 08:38:49|
+--------------------+--------------------+-------------------+
only showing top 3 rows



In [39]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)



# Products Dataset

In [18]:
products = spark.read.option('header', 'true').csv('data/input/unprocessed/olist_products_dataset.csv')

products.show(3)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [19]:
products = (products.select('product_id', 
                            'product_category_name', 
                            col('product_weight_g').cast(FloatType()),
                            col('product_length_cm').cast(FloatType()),
                            col('product_height_cm').cast(FloatType()),
                            col('product_width_cm').cast(FloatType())
                            )
            )


In [20]:
products.show(3)

+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|           225.0|             16.0|             10.0|            14.0|
|3aa071139cb16b67c...|                artes|          1000.0|             30.0|             18.0|            20.0|
|96bd76ec8810374ed...|        esporte_lazer|           154.0|             18.0|              9.0|            15.0|
+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
only showing top 3 rows



In [21]:
products_nulls = products.select([count(when(col(c).isNull(), c)).alias(c) for c in products.columns])
products_nulls.show()

+----------+---------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|               2|                2|                2|               2|
+----------+---------------------+----------------+-----------------+-----------------+----------------+



In [22]:
products_counts = products.select([count(col(c)).alias(c) for c in products.columns])


In [23]:
products_counts.show()

+----------+---------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+----------------+-----------------+-----------------+----------------+
|     32951|                32341|           32949|            32949|            32949|           32949|
+----------+---------------------+----------------+-----------------+-----------------+----------------+



In [24]:
products.count()

32951

In [25]:
filter_nulls = (products.filter(col('product_weight_g').isNull() | col('product_length_cm').isNull() | col('product_height_cm').isNull() | col('product_width_cm').isNull()))

filter_nulls.show()

+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+----------------+-----------------+-----------------+----------------+
|09ff539a621711667...|                bebes|            null|             null|             null|            null|
|5eb564652db742ff8...|                 null|            null|             null|             null|            null|
+--------------------+---------------------+----------------+-----------------+-----------------+----------------+



In [26]:
bebes = products.filter(col('product_category_name') == 'bebes')


In [27]:
bebes_measures = (
    bebes.select(
    mean('product_weight_g').alias('mean_weight'), 
    mean('product_length_cm').alias('mean_length'), 
    mean('product_height_cm').alias('mean_height'), 
    mean('product_width_cm').alias('mean_width')
    )
                  )

In [28]:
bebes_measures.show()

+------------------+-----------------+-----------------+-----------------+
|       mean_weight|      mean_length|      mean_height|       mean_width|
+------------------+-----------------+-----------------+-----------------+
|3655.2015250544664|37.14705882352941|21.61764705882353|28.71786492374728|
+------------------+-----------------+-----------------+-----------------+



In [29]:
bebes_means = bebes_measures.first()

products = products.fillna({
    'product_weight_g': bebes_means['mean_weight'],
    'product_length_cm': bebes_means['mean_length'],
    'product_height_cm': bebes_means['mean_height'],
    'product_width_cm': bebes_means['mean_width']
})

In [30]:
products.select([count(when(col(c).isNull(), c)).alias(c) for c in products.columns]).show()

+----------+---------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|               0|                0|                0|               0|
+----------+---------------------+----------------+-----------------+-----------------+----------------+



In [40]:
products.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_weight_g: float (nullable = false)
 |-- product_length_cm: float (nullable = false)
 |-- product_height_cm: float (nullable = false)
 |-- product_width_cm: float (nullable = false)



# Sellers Dataset

In [31]:
sellers = spark.read.option('header', 'true').csv('data/input/unprocessed/olist_sellers_dataset.csv')

sellers.show(3)

+--------------------+----------------------+--------------+------------+
|           seller_id|seller_zip_code_prefix|   seller_city|seller_state|
+--------------------+----------------------+--------------+------------+
|3442f8959a84dea7e...|                 13023|      campinas|          SP|
|d1b65fc7debc3361e...|                 13844|    mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|rio de janeiro|          RJ|
+--------------------+----------------------+--------------+------------+
only showing top 3 rows



In [32]:
sellers = sellers.withColumnRenamed('seller_zip_code_prefix', 'seller_zip_code')

In [33]:
sellers = sellers.withColumn('seller_zip_code', col('seller_zip_code').cast(IntegerType()))

In [34]:
sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [35]:
sellers.select([count(when(col(c).isNull(), c)).alias(c) for c in sellers.columns]).show()

+---------+---------------+-----------+------------+
|seller_id|seller_zip_code|seller_city|seller_state|
+---------+---------------+-----------+------------+
|        0|              0|          0|           0|
+---------+---------------+-----------+------------+



In [41]:
sellers.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)

